# Exponentially weighted moving average
The formula by hand, the effect of beta, the weights, the starting value, and pandas' `ewm`,
on the daily mean temperature of Delhi (2013-2016).

In [ ]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=4, suppress=True)

d = pd.read_csv("data/delhi_climate.csv", parse_dates=["date"])
theta = d.meantemp.to_numpy()
print(d.shape, "first day", d.date.iloc[0].date(), "last day", d.date.iloc[-1].date())
print("simple mean of all days:", theta.mean().round(2))

## The formula, step by step
V_t = beta * V_{t-1} + (1 - beta) * theta_t, worked on two days of 13 and 17 degrees.

In [ ]:
def ewma(values, beta, v0):
    """EWMA of a sequence: start from v0, then mix in each new value."""
    v, out = v0, []
    for t in values:
        v = beta * v + (1 - beta) * t
        out.append(v)
    return np.array(out)

print("V0 = 0:      ", ewma([13, 17], 0.9, v0=0.0))
print("V0 = theta1: ", ewma([13, 17], 0.9, v0=13.0))

## Unrolling the formula: older values get smaller weights
With V0 = 0, V4 = (1 - beta)(beta^3 theta1 + beta^2 theta2 + beta theta3 + theta4).

In [ ]:
beta = 0.9
t4 = theta[:4]
by_loop = ewma(t4, beta, v0=0.0)[-1]
by_sum = (1 - beta) * (beta**3 * t4[0] + beta**2 * t4[1] + beta * t4[2] + t4[3])
print("first four days:", t4)
print("V4 by the loop:", round(by_loop, 4), "  V4 by the unrolled sum:", round(by_sum, 4))
print("weights on theta1..theta4:", (1 - beta) * beta ** np.arange(3, -1, -1))

## About 1/(1 - beta) values: share of the total weight on the most recent 1/(1 - beta) points

In [ ]:
for beta in (0.5, 0.9, 0.98):
    n = round(1 / (1 - beta))
    print(f"beta {beta}: 1/(1-beta) = {n:3d} points carry {1 - beta**n:.3f} of the weight;"
          f" weight on the point {n} steps back = {(1 - beta) * beta**n:.4f} vs newest {(1 - beta):.2f}")

## The effect of beta: how spiky is the curve?
Mean absolute change of the EWMA from one day to the next (the data itself first).

In [ ]:
print("data itself:", np.abs(np.diff(theta)).mean().round(3))
for beta in (0.1, 0.5, 0.9, 0.98):
    v = ewma(theta, beta, v0=theta[0])
    print(f"beta {beta}: mean daily change {np.abs(np.diff(v)).mean():.3f}")

## The starting value V0

In [ ]:
v_zero = ewma(theta, 0.9, v0=0.0)
v_first = ewma(theta, 0.9, v0=theta[0])
print("day 1-5, V0 = 0:     ", v_zero[:5])
print("day 1-5, V0 = theta1:", v_first[:5])
print("data, day 1-5:        ", theta[:5])
gap = np.abs(v_zero - v_first)
print("gap on day 10, 20, 50:", gap[[9, 19, 49]].round(3), " first day with gap < 0.1:", int(np.argmax(gap < 0.1)) + 1)

## pandas: `ewm(alpha=1 - beta)`
`adjust=False` is our formula with V0 = theta1. The default `adjust=True` divides by the sum of the weights.

In [ ]:
pd_false = d.meantemp.ewm(alpha=0.1, adjust=False).mean().to_numpy()
pd_true = d.meantemp.ewm(alpha=0.1, adjust=True).mean().to_numpy()
print("adjust=False equals our loop with V0 = theta1:", np.allclose(pd_false, v_first))
t = np.arange(1, len(theta) + 1)
print("adjust=True equals V0 = 0 divided by (1 - beta^t):", np.allclose(pd_true, v_zero / (1 - 0.9**t)))
d["ewma"] = pd_false
print(d.head().round({"meantemp": 2, "ewma": 2}))